# Serrodyne Waveform Example

Generate a piecewise serrodyne sawtooth waveform with `firmware.signals.serrodyne`. The requested waveform duration determines the active DAC waveform length. `load_waveform()` zero-pads the BRAM write internally, then DAC0 is programmed to loop over only the active sample count.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from firmware import OverlayController
from firmware.signals import serrodyne

In [ ]:
ol = OverlayController()
info = ol.info()

DAC_SR = float(info["rfdc"]["dac0_sampling_rate_gsps"]) * 1e9
BUF_LEN = int(info["dac0"]["bram_int16_samples"])
DAC_PEAK = int(0.8 * np.iinfo(np.int16).max)
SAMPLES_PER_VECTOR = 512 // 16
VECTOR_COUNT = BUF_LEN // SAMPLES_PER_VECTOR

DAC_SR, BUF_LEN, DAC_PEAK, SAMPLES_PER_VECTOR, VECTOR_COUNT

In [ ]:
def to_int16(waveform, peak=DAC_PEAK):
    data = np.asarray(waveform, dtype=float)
    return np.round(np.clip(data, -float(peak), float(peak))).astype(np.int16)


def samples_for_duration(duration_s, sample_rate_hz, player):
    samples = int(round(float(duration_s) * float(sample_rate_hz)))
    if samples <= 0:
        raise ValueError("duration must produce at least one sample")
    if samples > player.capacity:
        raise ValueError(
            f"duration requires {samples} samples, but {player.name} capacity is {player.capacity}"
        )
    return samples


def duration_for_samples(num_samples, sample_rate_hz):
    return int(num_samples) / float(sample_rate_hz)


def vectors_for_samples(num_samples):
    return int(np.ceil(int(num_samples) / SAMPLES_PER_VECTOR))


def read_dac0_length_raw():
    return int(ol.gpio_control.axi_gpio_dac.mmio.read(0x8)) & ol.dac0.length_mask


def plot_time(waveform, sample_rate, samples=4096, title="Waveform"):
    view = np.asarray(waveform)[:samples]
    time_ns = np.arange(view.size) / sample_rate * 1e9
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(time_ns, view)
    ax.set_title(title)
    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("DAC code")
    ax.grid(True, alpha=0.3)
    return ax


def plot_eom_spectrum(phase_codes, sample_rate, title="Expected EOM spectrum"):
    data = np.asarray(phase_codes, dtype=float)
    span = float(np.max(data) - np.min(data))
    if span == 0.0:
        raise ValueError("Cannot plot spectrum for a constant waveform")
    phase = 2 * np.pi * data / span
    field = np.exp(1j * phase)
    spectrum = np.fft.fftshift(np.abs(np.fft.fft(field))) / len(field)
    freqs_mhz = np.fft.fftshift(np.fft.fftfreq(len(field), d=1 / sample_rate)) / 1e6

    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(freqs_mhz, spectrum)
    ax.set_title(title)
    ax.set_xlabel("Frequency (MHz)")
    ax.set_ylabel("Magnitude")
    ax.grid(True, alpha=0.3)
    return ax

## Generate Serrodyne Pattern

`requested_total_seconds` is converted to an integer active sample count. The overlay controller writes a full zero-padded BRAM buffer internally, while the hardware loop length is set to the active sample count.

In [ ]:
ratios = [1, 5, 3]
freqs_hz = [-1330e6, 0.0, 840e6]
requested_total_seconds = 1.0e-6

active_samples = samples_for_duration(requested_total_seconds, DAC_SR, ol.dac0)
actual_total_seconds = duration_for_samples(active_samples, DAC_SR)

x_base, y_base, n_base = serrodyne(
    ratios,
    freqs_hz,
    actual_total_seconds,
    amplitude=DAC_PEAK,
    sample_rate=DAC_SR,
    max_points=active_samples,
    continuous_phase=False,
)

active_waveform = to_int16(y_base)

print(f"DAC0 capacity: {ol.dac0.capacity} samples")
print(f"Requested serrodyne period: {requested_total_seconds * 1e6:.6f} us")
print(f"Active waveform length: {active_samples} samples")
print(f"Active vector count: {vectors_for_samples(active_samples)} vectors")
print(f"BRAM write length: {BUF_LEN} samples, padded internally")
print(f"Full BRAM vector count: {VECTOR_COUNT} vectors")
print(f"Actual serrodyne period: {actual_total_seconds * 1e6:.6f} us")

n_base, active_waveform.dtype, active_waveform.shape

In [ ]:
plot_time(active_waveform, DAC_SR, title="DAC0 active serrodyne waveform")
plot_eom_spectrum(active_waveform, DAC_SR, title="Expected serrodyne EOM spectrum");

## Program DAC0

Disable DAC0 explicitly before programming if output is currently active. `load_waveform()` zero-pads shorter waveforms and always writes the full BRAM buffer. `set_waveform_length(active_samples)` sets the active hardware loop length. The raw GPIO readback should match `active_samples` before enabling DAC0.

In [ ]:
print("Before programming")
print(f"  active_samples: {active_samples} ({active_samples:#x})")
print(f"  active vectors: {vectors_for_samples(active_samples)}")
print(f"  BRAM samples to write: {BUF_LEN} ({BUF_LEN:#x}), padded internally")
print(f"  raw length before: {read_dac0_length_raw()} ({read_dac0_length_raw():#x})")

ol.dac0.disable()
time.sleep(0.01)
ol.dac0.load_waveform(active_waveform)
print(f"Raw length after BRAM load: {read_dac0_length_raw()} ({read_dac0_length_raw():#x})")

time.sleep(0.01)
length_readback = read_dac0_length_raw()
print(f"Raw active length readback: {length_readback} ({length_readback:#x})")

if length_readback != active_samples:
    raise RuntimeError(
        f"DAC0 length mismatch: wrote {active_samples:#x}, read {length_readback:#x}"
    )

ol.dac0.info()

## Enable and Disable DAC0

In [ ]:
ol.dac0.enable()
ol.dac0.is_enabled()

In [ ]:
ol.dac0.disable()
ol.info()